# MICS DDML analysis

This notebook prepares the household and child samples, evaluates treatment support, estimates the DoubleML models, and builds the reported diagnostics and robustness results.

The notebook is self-contained. Fitted models and expensive out-of-fold predictions are checkpointed automatically, so a compatible result is loaded instead of being estimated again.


## Setup

### Project paths


In [1]:
from pathlib import Path
import os
import pickle
from warnings import filterwarnings


ROOT = Path("../../").resolve()
DATA = ROOT / "Data" / "3. Final"
OUT = ROOT / "Output"
FIGS = ROOT / "Figures"
TABLES = ROOT / "Writing edit" / "Table"

MODELS = OUT / "models" / "grouped_convex_sl"
OOF_MODELS = MODELS / "oof_propensities"
SUPPORT_MODELS = MODELS / "support_restricted"
LOCO_MODELS = MODELS / "leave_one_country_out"
ROBUSTNESS_MODELS = MODELS / "nuisance_learner_robustness"

for folder in [
    OUT,
    FIGS,
    TABLES,
    MODELS,
    OOF_MODELS,
    SUPPORT_MODELS,
    LOCO_MODELS,
    ROBUSTNESS_MODELS,
]:
    folder.mkdir(parents=True, exist_ok=True)

HH_FILE = DATA / "MASTER_MICS_FINAL.dta"
U5_FILE = DATA / "MASTER_MICS_FINAL_U5.dta"

if not HH_FILE.is_file() or not U5_FILE.is_file():
    raise FileNotFoundError("The HH or U5 source file is missing.")

filterwarnings("ignore")

for variable in [
    "OMP_NUM_THREADS",
    "MKL_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "NUMEXPR_NUM_THREADS",
]:
    os.environ[variable] = "1"

os.environ["MPLCONFIGDIR"] = "/tmp/mics_ddml_matplotlib"
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

print(ROOT)


/home/jadrk040507/Dropbox/MICS_DDML


### Imports


In [2]:
import doubleml as dml
import matplotlib.pyplot as plt
import narwhals._interchange
import numpy as np
import pandas as pd
import pyreadstat
import sklearn

from IPython.display import display
from joblib import hash as joblib_hash
from scipy.optimize import minimize
from sklearn.base import (
    BaseEstimator,
    ClassifierMixin,
    RegressorMixin,
    clone,
)
from sklearn.ensemble import (
    RandomForestClassifier,
    RandomForestRegressor,
)
from sklearn.linear_model import (
    ElasticNetCV,
    LassoCV,
    LinearRegression,
    LogisticRegression,
    LogisticRegressionCV,
    RidgeCV,
)
from sklearn.model_selection import (
    GroupKFold,
    StratifiedGroupKFold,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm
from xgboost import XGBClassifier, XGBRegressor

try:
    from doubleml.utils import PSProcessorConfig
except ImportError:
    PSProcessorConfig = None


### Runtime and model settings


In [3]:
SEED = 42
FOLDS = 5
IRM_REPS = 3
APOS_REPS = 3
ROBUSTNESS_REPS = 1
LOCO_REPS = 1
TRIM = 0.01

LEVELS = {
    0: "No treatment",
    1: "Boiling",
    2: "Chlorination/tablets",
    3: "Straining/settling",
}

OUTCOME_LABELS = {
    "SomeRiskHome": "Any detectable E. coli at home",
    "VeryHighRiskHome": "Very high E. coli at home (>100 CFU/100 mL)",
    "diarrhea": "Diarrhea among children under five",
}

CPU = os.cpu_count() or 1
WORKERS = max(1, CPU - 1)
IRM_WORKERS = min(WORKERS, FOLDS)
APOS_WORKERS = min(WORKERS, len(LEVELS))
TUNING_WORKERS = max(1, WORKERS // max(IRM_WORKERS, APOS_WORKERS))

settings = pd.DataFrame(
    {
        "available_cpus": [CPU],
        "irm_fold_workers": [IRM_WORKERS],
        "apos_level_workers": [APOS_WORKERS],
        "inner_tuning_workers": [TUNING_WORKERS],
        "folds": [FOLDS],
        "irm_repetitions": [IRM_REPS],
        "apos_repetitions": [APOS_REPS],
        "propensity_clipping": [TRIM],
    }
)

display(settings)


,available_cpus,irm_fold_workers,apos_level_workers,inner_tuning_workers,folds,irm_repetitions,apos_repetitions,propensity_clipping
0,16,5,4,3,5,3,3,0.01


## Data

### Outcome variables

`SomeRiskHome` equals one whenever E. coli is detectable at the household. It therefore includes every observation classified as `VeryHighRiskHome`. The latter equals one only when the measured concentration exceeds 100 CFU per 100 mL.


In [4]:
def create_outcomes(df):
    '''Create nested E. coli outcomes while preserving missing values.'''

    df = df.copy()
    risk = pd.to_numeric(df["RiskHome"], errors="coerce")
    missing = risk.isna()

    df["SomeRiskHome"] = (
        risk.isin([1, 2])
        .mask(missing)
        .astype("Int8")
    )

    df["VeryHighRiskHome"] = (
        risk.eq(2)
        .mask(missing)
        .astype("Int8")
    )

    inconsistent = (
        df["VeryHighRiskHome"].eq(1)
        & ~df["SomeRiskHome"].eq(1)
    )

    assert not inconsistent.any()
    assert df["SomeRiskHome"].isna().equals(
        df["VeryHighRiskHome"].isna()
    )

    return df


### Water-treatment variables

The names follow the Stata cleaning file:

| Code | Treatment category |
|---:|---|
| 0 | No treatment |
| 1 | Boiling |
| 2 | Chlorination/tablets |
| 3 | Straining/settling |

When several methods are reported, boiling takes precedence, followed by chlorination/tablets and then straining/settling. A recognized method also takes precedence over `Other`. Households reporting only `Other` are removed rather than recoded as untreated.

Solar treatment remains part of the binary **any recognized treatment** comparison, but it is not estimated as a separate treatment category in the method-specific analysis.


In [5]:
def selected(series, letter):
    '''Return whether a MICS multiple-response item selected its letter.'''

    return (
        series.astype("string")
        .str.strip()
        .str.upper()
        .eq(letter.upper())
    )


def create_treatments(df):
    '''Create binary and categorical treatment variables.'''

    df = df.copy()

    method_cols = [
        "WQ15A",
        "WQ15B",
        "WQ15C",
        "WQ15D",
        "WQ15E",
        "WQ15F",
        "WQ15G",
        "WQ15H",
        "WQ15X",
    ]

    required = method_cols + ["WQ15_g"]
    missing = sorted(set(required) - set(df.columns))

    if missing:
        raise KeyError(f"Missing water-treatment variables: {missing}")

    used = pd.DataFrame(
        {
            column: selected(df[column], column[-1])
            for column in method_cols
        },
        index=df.index,
    )

    boiling = used["WQ15A"]
    chlorination = (
        used["WQ15B"]
        | used["WQ15G"]
        | used["WQ15H"]
    )
    straining_settling = (
        used["WQ15C"]
        | used["WQ15D"]
        | used["WQ15F"]
    )
    solar = used["WQ15E"]
    other = used["WQ15X"]

    recognized = (
        boiling
        | chlorination
        | straining_settling
        | solar
    )

    response_code = pd.to_numeric(df["WQ15_g"], errors="coerce")
    response_missing = (
        response_code.isna()
        | response_code.isin([99, 998])
    ) & ~used.any(axis=1)

    other_only = other & ~recognized
    no_treatment = ~used.any(axis=1) & ~response_missing

    category = pd.Series(pd.NA, index=df.index, dtype="Int8")
    category.loc[no_treatment] = 0
    category.loc[straining_settling] = 3
    category.loc[chlorination] = 2
    category.loc[boiling] = 1

    any_treatment = pd.Series(pd.NA, index=df.index, dtype="Int8")
    any_treatment.loc[no_treatment] = 0
    any_treatment.loc[recognized] = 1

    df["treatment_count"] = used.sum(axis=1).astype("int16")
    df["multiple_methods"] = df["treatment_count"].gt(1)
    df["other_only"] = other_only
    df["solar_treatment"] = solar
    df["water_treatment"] = any_treatment
    df["treat_cat"] = category

    df["treat_boil"] = (
        category.eq(1)
        .fillna(False)
        .astype("int8")
    )
    df["treat_chlorination_tablets"] = (
        category.eq(2)
        .fillna(False)
        .astype("int8")
    )
    df["treat_straining_settling"] = (
        category.eq(3)
        .fillna(False)
        .astype("int8")
    )

    diagnostics = {
        "rows_before_other_drop": len(df),
        "multiple_method_rows": int(df["multiple_methods"].sum()),
        "multiple_method_share": float(df["multiple_methods"].mean()*100),
        "other_only_rows_dropped": int(other_only.sum()),
        "missing_treatment_rows": int(response_missing.sum()),
        "solar_rows": int(solar.sum()),
    }

    df = df.loc[~other_only].copy()

    assert not df["other_only"].any()
    assert set(df["treat_cat"].dropna().unique()).issubset(LEVELS)

    return df, diagnostics


### Survey identifiers

A PSU is treated as one indivisible cross-fitting group. The raw PSU code is combined with country only when the same code appears in more than one country. Household identifiers are checked in the same way.


In [6]:
def coalesce_columns(df, names):
    '''Use the first observed value among a list of columns.'''

    result = pd.Series(pd.NA, index=df.index, dtype="object")
    found = []

    for name in names:
        if name in df.columns:
            found.append(name)
            result = result.where(result.notna(), df[name])

    if not found:
        raise KeyError(f"None of these columns is available: {names}")

    return result, found


def factorize_key(frame):
    '''Create a stable integer identifier from one or more columns.'''

    clean = frame.astype("string").fillna("<missing>")
    key = pd.MultiIndex.from_frame(clean)
    return pd.factorize(key, sort=True)[0].astype("int64")


In [7]:
def create_design_ids(df):
    '''Create globally valid PSU and household identifiers.'''

    df = df.copy()
    country = df["country_cat"].astype("string")

    psu_raw, psu_sources = coalesce_columns(
        df,
        ["PSU", "psu", "HH1"],
    )

    if psu_raw.isna().any():
        raise ValueError("Some observations have no valid PSU identifier.")

    psu_country_count = (
        pd.DataFrame(
            {
                "psu": psu_raw.astype("string"),
                "country": country,
            }
        )
        .groupby("psu")["country"]
        .nunique()
    )

    psu_reused = bool((psu_country_count > 1).any())

    if psu_reused:
        psu_key = pd.DataFrame(
            {
                "country": country,
                "psu": psu_raw,
            }
        )
    else:
        psu_key = pd.DataFrame({"psu": psu_raw})

    df["_psu_id"] = factorize_key(psu_key)

    if "HHID" in df.columns:
        household_raw = df["HHID"]
        household_sources = ["HHID"]
    elif {"HH1", "HH2"}.issubset(df.columns):
        household_raw = (
            df["HH1"].astype("string")
            + "|"
            + df["HH2"].astype("string")
        )
        household_sources = ["HH1", "HH2"]
    else:
        household_raw, household_sources = coalesce_columns(
            df,
            ["HH2", "HH1"],
        )

    if household_raw.isna().any():
        raise ValueError("Some observations have no valid household identifier.")

    household_country_count = (
        pd.DataFrame(
            {
                "household": household_raw.astype("string"),
                "country": country,
            }
        )
        .groupby("household")["country"]
        .nunique()
    )

    household_reused = bool((household_country_count > 1).any())

    if household_reused:
        household_key = pd.DataFrame(
            {
                "country": country,
                "household": household_raw,
            }
        )
    else:
        household_key = pd.DataFrame(
            {"household": household_raw}
        )

    df["_hh_id"] = factorize_key(household_key)

    diagnostics = {
        "psu_source_columns": ", ".join(psu_sources),
        "psu_reused_across_countries": psu_reused,
        "household_source_columns": ", ".join(household_sources),
        "household_reused_across_countries": household_reused,
    }

    return df, diagnostics


### Household and child samples


In [8]:
def prepare_sample(raw, child=False):
    '''Prepare the HH or U5 sample before constructing model controls.'''

    required = [
        "country_cat",
        "RiskHome",
        "windex5",
        "urban",
        "WS1_g",
        "wq27_decile",
        "Any_U5",
        "Girls_less_than15",
        "Boys_15or_less",
        "Toilet",
        "HHCHILDREN",
    ]

    missing = sorted(set(required) - set(raw.columns))

    if missing:
        raise KeyError(f"Missing required variables: {missing}")

    df = raw.copy()
    df["_row_id"] = np.arange(len(df), dtype="int64")

    df = create_outcomes(df)
    df, treatment_diagnostics = create_treatments(df)
    df, identifier_diagnostics = create_design_ids(df)

    for column in [
        "Any_U5",
        "Girls_less_than15",
        "Boys_15or_less",
    ]:
        df[column] = (
            pd.to_numeric(df[column], errors="coerce")
            .fillna(0)
            .astype("int8")
        )

    df["num_children"] = (
        pd.to_numeric(df["HHCHILDREN"], errors="coerce")
        .fillna(0)
        .clip(upper=10)
    )

    if child:
        child_variables = ["age", "male", "diarrhea"]
        missing = sorted(set(child_variables) - set(df.columns))

        if missing:
            raise KeyError(f"Missing U5 variables: {missing}")

        df["child_age"] = pd.to_numeric(
            df["age"],
            errors="coerce",
        )
        df["child_sex_male"] = pd.to_numeric(
            df["male"],
            errors="coerce",
        )
        df["diarrhea"] = (
            pd.to_numeric(df["diarrhea"], errors="coerce")
            .astype("Int8")
        )

    diagnostics = {
        **treatment_diagnostics,
        **identifier_diagnostics,
    }

    return df, diagnostics


In [9]:
hh_raw, _ = pyreadstat.read_dta(HH_FILE)
u5_raw, _ = pyreadstat.read_dta(U5_FILE)

hh, hh_cleaning = prepare_sample(hh_raw)
u5, u5_cleaning = prepare_sample(u5_raw, child=True)

cleaning_diagnostics = pd.DataFrame(
    [
        {"sample": "HH", **hh_cleaning},
        {"sample": "U5", **u5_cleaning},
    ]
)

cleaning_diagnostics.to_csv(
    OUT / "cleaning_and_identifier_diagnostics.csv",
    index=False,
)

display(cleaning_diagnostics)


,sample,rows_before_other_drop,multiple_method_rows,multiple_method_share,other_only_rows_dropped,missing_treatment_rows,solar_rows,psu_source_columns,psu_reused_across_countries,household_source_columns,household_reused_across_countries
0,HH,59620,1594,2.673599,146,0,21,"PSU, psu, HH1",True,"HH1, HH2",True
1,U5,36121,695,1.924088,97,0,9,"PSU, psu, HH1",True,HHID,False


### Controls

Country indicators are rebuilt within every estimation sample. This is especially important for leave-one-country-out models.


In [10]:
HH_CONTROL_VARIABLES = [
    "windex5",
    "urban",
    "WS1_g",
    "wq27_decile",
    "Any_U5",
    "Girls_less_than15",
    "Boys_15or_less",
    "Toilet",
]

U5_CONTROL_VARIABLES = [
    *HH_CONTROL_VARIABLES,
    "child_age",
    "child_sex_male",
]


def dummy_block(series, prefix, drop_first=True):
    '''Create a complete dummy block with a clear prefix.'''

    clean = series.astype("string")

    return pd.get_dummies(
        clean,
        prefix=prefix,
        drop_first=drop_first,
        dtype=float,
    )


In [11]:
def build_controls(df, child=False):
    '''Construct the model matrix from the raw analysis sample.'''

    blocks = [
        dummy_block(df["windex5"], "wealth"),
        dummy_block(df["country_cat"], "country"),
        dummy_block(df["WS1_g"], "water_source"),
        dummy_block(df["Toilet"], "toilet"),
        dummy_block(df["wq27_decile"], "source_ecoli"),
        df[[
            "urban",
            "Any_U5",
            "Girls_less_than15",
            "Boys_15or_less",
        ]].astype(float),
    ]

    if child:
        blocks.extend(
            [
                dummy_block(df["child_age"], "child_age"),
                df[["child_sex_male"]].astype(float),
            ]
        )

    controls = pd.concat(blocks, axis=1)
    controls = controls.loc[:, ~controls.columns.duplicated()]

    return controls.astype(float)


In [12]:
def analysis_frame(
    df,
    outcome,
    treatment,
    child=False,
    allowed_levels=None,
):
    '''Build one complete-case estimation frame and its control list.'''

    raw_controls = (
        U5_CONTROL_VARIABLES
        if child
        else HH_CONTROL_VARIABLES
    )

    required = [
        outcome,
        treatment,
        "country_cat",
        "_psu_id",
        "_hh_id",
        *raw_controls,
    ]

    keep = df[required].notna().all(axis=1)

    if allowed_levels is not None:
        keep &= df[treatment].isin(allowed_levels)

    sample = df.loc[keep].copy()
    controls = build_controls(sample, child=child)

    frame = pd.concat(
        [
            sample[[
                "_row_id",
                "country_cat",
                "_psu_id",
                "_hh_id",
                outcome,
                treatment,
            ]],
            controls,
        ],
        axis=1,
    )

    frame[outcome] = pd.to_numeric(
        frame[outcome],
        errors="raise",
    ).astype(float)

    frame[treatment] = pd.to_numeric(
        frame[treatment],
        errors="raise",
    ).astype(int)

    frame["_psu_model_code"] = pd.factorize(
        frame["_psu_id"],
        sort=True,
    )[0].astype(float)

    x_cols = [
        *controls.columns.tolist(),
        "_psu_model_code",
    ]

    return frame.reset_index(drop=True), x_cols


## Model specification

### Base learners

The individual learners form the Super Learner library. Their causal estimates are not treated as independent estimators in the main results; the publication specification uses the convex Super Learner.


In [13]:
def base_learner_library():
    '''Return the outcome and propensity learner libraries.'''

    grid = np.logspace(-3, 3, 7)

    outcome_learners = {
        "ols": LinearRegression(),
        "lasso": Pipeline(
            [
                ("scale", StandardScaler()),
                (
                    "model",
                    LassoCV(
                        cv=3,
                        max_iter=5_000,
                        n_jobs=1,
                        random_state=SEED,
                    ),
                ),
            ]
        ),
        "ridge": Pipeline(
            [
                ("scale", StandardScaler()),
                (
                    "model",
                    RidgeCV(
                        alphas=grid,
                        cv=3,
                    ),
                ),
            ]
        ),
        "enet": Pipeline(
            [
                ("scale", StandardScaler()),
                (
                    "model",
                    ElasticNetCV(
                        cv=3,
                        l1_ratio=[0.5],
                        max_iter=5_000,
                        n_jobs=1,
                        random_state=SEED,
                    ),
                ),
            ]
        ),
        "rf": RandomForestRegressor(
            n_estimators=200,
            max_depth=15,
            min_samples_leaf=5,
            random_state=SEED,
            n_jobs=1,
        ),
        "xgb": XGBRegressor(
            n_estimators=150,
            max_depth=4,
            learning_rate=0.1,
            subsample=0.8,
            random_state=SEED,
            n_jobs=1,
            eval_metric="rmse",
        ),
    }

    propensity_learners = {
        "logit": Pipeline(
            [
                ("scale", StandardScaler()),
                (
                    "model",
                    LogisticRegression(
                        penalty=None,
                        solver="lbfgs",
                        max_iter=2_000,
                    ),
                ),
            ]
        ),
        "lasso": Pipeline(
            [
                ("scale", StandardScaler()),
                (
                    "model",
                    LogisticRegressionCV(
                        Cs=grid,
                        cv=3,
                        penalty="l1",
                        solver="liblinear",
                        scoring="neg_log_loss",
                        max_iter=2_000,
                        n_jobs=1,
                        random_state=SEED,
                    ),
                ),
            ]
        ),
        "ridge": Pipeline(
            [
                ("scale", StandardScaler()),
                (
                    "model",
                    LogisticRegressionCV(
                        Cs=grid,
                        cv=3,
                        penalty="l2",
                        solver="lbfgs",
                        scoring="neg_log_loss",
                        max_iter=2_000,
                        n_jobs=1,
                        random_state=SEED,
                    ),
                ),
            ]
        ),
        "enet": Pipeline(
            [
                ("scale", StandardScaler()),
                (
                    "model",
                    LogisticRegressionCV(
                        Cs=np.logspace(-2, 2, 5),
                        cv=3,
                        penalty="elasticnet",
                        solver="saga",
                        l1_ratios=[0.5],
                        scoring="neg_log_loss",
                        max_iter=3_000,
                        n_jobs=1,
                        random_state=SEED,
                    ),
                ),
            ]
        ),
        "rf": RandomForestClassifier(
            n_estimators=200,
            max_depth=15,
            min_samples_leaf=5,
            random_state=SEED,
            n_jobs=1,
        ),
        "xgb": XGBClassifier(
            n_estimators=150,
            max_depth=4,
            learning_rate=0.1,
            subsample=0.8,
            random_state=SEED,
            n_jobs=1,
            eval_metric="logloss",
        ),
    }

    return outcome_learners, propensity_learners


REG_LEARNERS, CLF_LEARNERS = base_learner_library()


### Convex Super Learner

The meta-learner chooses nonnegative weights that sum to one. Outcome weights minimize out-of-fold squared error, while propensity weights minimize out-of-fold Bernoulli log loss.

The final column of each model matrix contains the PSU identifier used only for inner grouped validation. It is removed before fitting the base learners, so PSU codes are never used as predictors.


In [14]:
def split_features_and_groups(X, group_column=-1):
    '''Separate model features from the PSU code carried with X.'''

    X = np.asarray(X)
    groups = X[:, group_column].astype("int64")
    features = np.delete(X, group_column, axis=1)

    return features, groups


def inner_grouped_splits(
    y,
    groups,
    n_splits,
    seed,
    classification,
):
    '''Construct grouped inner folds for Super Learner weights.'''

    y = np.asarray(y)
    groups = np.asarray(groups)
    maximum = min(n_splits, np.unique(groups).size)

    for folds in range(maximum, 1, -1):
        if classification:
            splitter = StratifiedGroupKFold(
                n_splits=folds,
                shuffle=True,
                random_state=seed,
            )
            candidate = list(
                splitter.split(
                    np.zeros(len(y)),
                    y,
                    groups,
                )
            )

            valid = all(
                np.unique(y[train]).size == 2
                for train, _ in candidate
            )
        else:
            splitter = GroupKFold(n_splits=folds)
            candidate = list(
                splitter.split(
                    np.zeros(len(y)),
                    y,
                    groups,
                )
            )
            valid = True

        if valid:
            return candidate

    raise ValueError(
        "The training sample does not contain enough PSU variation "
        "for grouped inner validation."
    )


In [15]:
def convex_weights(predictions, target, loss):
    '''Estimate nonnegative weights that sum to one.'''

    predictions = np.asarray(predictions, dtype=float)
    target = np.asarray(target, dtype=float)
    n_learners = predictions.shape[1]

    def objective(weights):
        combined = predictions @ weights

        if loss == "mse":
            return np.mean((target - combined) ** 2)

        combined = np.clip(combined, 1e-6, 1 - 1e-6)
        return -np.mean(
            target * np.log(combined)
            + (1 - target) * np.log(1 - combined)
        )

    initial = np.repeat(1 / n_learners, n_learners)

    result = minimize(
        objective,
        initial,
        method="SLSQP",
        bounds=[(0, 1)] * n_learners,
        constraints={
            "type": "eq",
            "fun": lambda weights: weights.sum() - 1,
        },
    )

    if result.success:
        weights = np.clip(result.x, 0, 1)
        weights = weights / weights.sum()
        return weights

    single_losses = []

    for column in range(n_learners):
        candidate = np.zeros(n_learners)
        candidate[column] = 1
        single_losses.append(objective(candidate))

    weights = np.zeros(n_learners)
    weights[int(np.argmin(single_losses))] = 1

    return weights


In [16]:
class ConvexSuperLearnerRegressor(RegressorMixin, BaseEstimator):
    '''Convex Super Learner for an outcome regression.'''

    def __init__(
        self,
        estimators,
        n_folds=3,
        random_state=42,
        group_column=-1,
    ):
        self.estimators = estimators
        self.n_folds = n_folds
        self.random_state = random_state
        self.group_column = group_column

    def fit(self, X, y):
        features, groups = split_features_and_groups(
            X,
            self.group_column,
        )
        target = np.asarray(y, dtype=float)

        splits = inner_grouped_splits(
            target,
            groups,
            self.n_folds,
            self.random_state,
            classification=False,
        )

        oof = np.full(
            (len(target), len(self.estimators)),
            np.nan,
        )

        for column, (_, estimator) in enumerate(self.estimators):
            for train, test in splits:
                fitted = clone(estimator)
                fitted.fit(features[train], target[train])
                oof[test, column] = fitted.predict(features[test])

        if np.isnan(oof).any():
            raise RuntimeError("Incomplete Super Learner outcome predictions.")

        self.weights_ = convex_weights(oof, target, loss="mse")
        self.estimators_ = []

        for name, estimator in self.estimators:
            fitted = clone(estimator)
            fitted.fit(features, target)
            self.estimators_.append((name, fitted))

        self.estimator_names_ = [name for name, _ in self.estimators]
        self.n_features_in_ = np.asarray(X).shape[1]

        return self

    def predict(self, X):
        features, _ = split_features_and_groups(
            X,
            self.group_column,
        )

        predictions = np.column_stack(
            [
                estimator.predict(features)
                for _, estimator in self.estimators_
            ]
        )

        return predictions @ self.weights_


In [17]:
class ConvexSuperLearnerClassifier(ClassifierMixin, BaseEstimator):
    '''Convex probability ensemble for the propensity score.'''

    def __init__(
        self,
        estimators,
        n_folds=3,
        random_state=42,
        group_column=-1,
    ):
        self.estimators = estimators
        self.n_folds = n_folds
        self.random_state = random_state
        self.group_column = group_column

    def fit(self, X, y):
        features, groups = split_features_and_groups(
            X,
            self.group_column,
        )
        target = np.asarray(y, dtype=int)

        if np.unique(target).size != 2:
            raise ValueError("The propensity learner requires two classes.")

        splits = inner_grouped_splits(
            target,
            groups,
            self.n_folds,
            self.random_state,
            classification=True,
        )

        oof = np.full(
            (len(target), len(self.estimators)),
            np.nan,
        )

        for column, (_, estimator) in enumerate(self.estimators):
            for train, test in splits:
                fitted = clone(estimator)
                fitted.fit(features[train], target[train])
                oof[test, column] = fitted.predict_proba(
                    features[test]
                )[:, 1]

        if np.isnan(oof).any():
            raise RuntimeError("Incomplete Super Learner propensity predictions.")

        self.weights_ = convex_weights(
            oof,
            target,
            loss="log_loss",
        )
        self.estimators_ = []

        for name, estimator in self.estimators:
            fitted = clone(estimator)
            fitted.fit(features, target)
            self.estimators_.append((name, fitted))

        self.estimator_names_ = [name for name, _ in self.estimators]
        self.classes_ = np.array([0, 1])
        self.n_features_in_ = np.asarray(X).shape[1]

        return self

    def predict_proba(self, X):
        features, _ = split_features_and_groups(
            X,
            self.group_column,
        )

        probabilities = np.column_stack(
            [
                estimator.predict_proba(features)[:, 1]
                for _, estimator in self.estimators_
            ]
        )

        positive = np.clip(
            probabilities @ self.weights_,
            1e-8,
            1 - 1e-8,
        )

        return np.column_stack([1 - positive, positive])

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)


In [18]:
ML_G = ConvexSuperLearnerRegressor(
    estimators=list(REG_LEARNERS.items()),
    n_folds=3,
    random_state=SEED,
)

ML_M = ConvexSuperLearnerClassifier(
    estimators=list(CLF_LEARNERS.items()),
    n_folds=3,
    random_state=SEED,
)


### Analysis samples


In [19]:
hh_some_irm, hh_irm_x = analysis_frame(
    hh,
    outcome="SomeRiskHome",
    treatment="water_treatment",
)

hh_vhigh_irm, hh_vhigh_irm_x = analysis_frame(
    hh,
    outcome="VeryHighRiskHome",
    treatment="water_treatment",
)

hh_some_apos, hh_apos_x = analysis_frame(
    hh,
    outcome="SomeRiskHome",
    treatment="treat_cat",
    allowed_levels=list(LEVELS),
)

hh_vhigh_apos, hh_vhigh_apos_x = analysis_frame(
    hh,
    outcome="VeryHighRiskHome",
    treatment="treat_cat",
    allowed_levels=list(LEVELS),
)

u5_irm, u5_irm_x = analysis_frame(
    u5,
    outcome="diarrhea",
    treatment="water_treatment",
    child=True,
)

u5_apos, u5_apos_x = analysis_frame(
    u5,
    outcome="diarrhea",
    treatment="treat_cat",
    child=True,
    allowed_levels=list(LEVELS),
)


In [20]:
assert hh_some_irm["_row_id"].equals(
    hh_vhigh_irm["_row_id"]
)
assert hh_some_apos["_row_id"].equals(
    hh_vhigh_apos["_row_id"]
)
assert hh_irm_x == hh_vhigh_irm_x
assert hh_apos_x == hh_vhigh_apos_x

analysis_samples = pd.DataFrame(
    [
        {
            "sample": "HH binary",
            "observations": len(hh_some_irm),
            "PSUs": hh_some_irm["_psu_id"].nunique(),
            "households": hh_some_irm["_hh_id"].nunique(),
        },
        {
            "sample": "HH treatment categories",
            "observations": len(hh_some_apos),
            "PSUs": hh_some_apos["_psu_id"].nunique(),
            "households": hh_some_apos["_hh_id"].nunique(),
        },
        {
            "sample": "U5 binary",
            "observations": len(u5_irm),
            "PSUs": u5_irm["_psu_id"].nunique(),
            "households": u5_irm["_hh_id"].nunique(),
        },
        {
            "sample": "U5 treatment categories",
            "observations": len(u5_apos),
            "PSUs": u5_apos["_psu_id"].nunique(),
            "households": u5_apos["_hh_id"].nunique(),
        },
    ]
)

display(analysis_samples)


,sample,observations,PSUs,households
0,HH binary,59474,18650,59474
1,HH treatment categories,59463,18649,59463
2,U5 binary,36024,13583,25136
3,U5 treatment categories,36016,13582,25132


## Pre-estimation diagnostics

### Treatment frequencies


In [21]:
def treatment_frequencies(frame, treatment, sample_name):
    '''Count observations and PSU by treatment category.'''

    table = (
        frame.groupby(treatment)
        .agg(
            observations=("_row_id", "size"),
            PSUs=("_psu_id", "nunique"),
            households=("_hh_id", "nunique"),
        )
        .reset_index()
        .rename(columns={treatment: "treatment_code"})
    )

    table["sample"] = sample_name
    table["treatment_category"] = table["treatment_code"].map(LEVELS)
    table["share"] = table["observations"] / table["observations"].sum()

    return table[
        [
            "sample",
            "treatment_code",
            "treatment_category",
            "observations",
            "PSUs",
            "households",
            "share",
        ]
    ]


treatment_table = pd.concat(
    [
        treatment_frequencies(
            hh_some_apos,
            "treat_cat",
            "HH",
        ),
        treatment_frequencies(
            u5_apos,
            "treat_cat",
            "U5",
        ),
    ],
    ignore_index=True,
)

treatment_table.to_csv(
    OUT / "treatment_category_frequencies.csv",
    index=False,
)

display(treatment_table)


,sample,treatment_code,treatment_category,observations,PSUs,households,share
0,HH,0,No treatment,46951,16973,46951,0.789583
1,HH,1,Boiling,7238,3070,7238,0.121723
2,HH,2,Chlorination/tablets,1458,1163,1458,0.024519
3,HH,3,Straining/settling,3816,2115,3816,0.064174
4,U5,0,No treatment,29097,11567,20247,0.807891
5,U5,1,Boiling,3216,1750,2468,0.089294
6,U5,2,Chlorination/tablets,838,521,593,0.023267
7,U5,3,Straining/settling,2865,1145,1824,0.079548


### PSU and household structure

The following table reports the number of PSU and the mean, median, upper tail, and maximum number of observations per PSU. For the U5 sample, it also reveals repeated children within households.


In [22]:
def structure_summary(frame, sample_name):
    '''Summarize observations within PSU and households.'''

    psu_sizes = frame.groupby("_psu_id").size()
    household_sizes = frame.groupby("_hh_id").size()

    return {
        "sample": sample_name,
        "observations": len(frame),
        "PSUs": frame["_psu_id"].nunique(),
        "households": frame["_hh_id"].nunique(),
        "mean_observations_per_PSU": psu_sizes.mean(),
        "median_observations_per_PSU": psu_sizes.median(),
        "p90_observations_per_PSU": psu_sizes.quantile(0.90),
        "maximum_observations_per_PSU": psu_sizes.max(),
        "mean_observations_per_household": household_sizes.mean(),
        "median_observations_per_household": household_sizes.median(),
    }


structure_table = pd.DataFrame(
    [
        structure_summary(hh_some_irm, "HH binary"),
        structure_summary(hh_some_apos, "HH treatment categories"),
        structure_summary(u5_irm, "U5 binary"),
        structure_summary(u5_apos, "U5 treatment categories"),
    ]
)

structure_table.to_csv(
    OUT / "psu_structure_summary.csv",
    index=False,
)

display(structure_table)


,sample,observations,PSUs,households,mean_observations_per_PSU,median_observations_per_PSU,p90_observations_per_PSU,maximum_observations_per_PSU,mean_observations_per_household,median_observations_per_household
0,HH binary,59474,18650,59474,3.188954,3.0,5.0,8,1.000000,1.0
1,HH treatment categories,59463,18649,59463,3.188536,3.0,5.0,8,1.000000,1.0
2,U5 binary,36024,13583,25136,2.652139,2.0,5.0,25,1.433164,1.0
3,U5 treatment categories,36016,13582,25132,2.651745,2.0,5.0,25,1.433073,1.0


### Observed support by country and PSU

For each comparison, the basic condition is that both `No treatment` and the relevant treatment category occur within the country. The stricter diagnostic asks whether each category occurs in at least two distinct PSU. It does not require every individual PSU to contain both categories.


In [23]:
def support_by_country(frame, sample_name):
    '''Evaluate observed category support within each country.'''

    counts = (
        frame.groupby(
            ["country_cat", "treat_cat"],
            dropna=False,
        )
        .agg(
            observations=("_row_id", "size"),
            PSUs=("_psu_id", "nunique"),
            households=("_hh_id", "nunique"),
        )
        .reset_index()
    )

    countries = sorted(frame["country_cat"].dropna().unique())
    detail_rows = []
    summary_rows = []

    for level in [1, 2, 3]:
        comparison_rows = []

        for country in countries:
            country_counts = (
                counts.loc[counts["country_cat"].eq(country)]
                .set_index("treat_cat")
            )

            n_control = (
                int(country_counts.loc[0, "observations"])
                if 0 in country_counts.index
                else 0
            )
            psu_control = (
                int(country_counts.loc[0, "PSUs"])
                if 0 in country_counts.index
                else 0
            )
            n_category = (
                int(country_counts.loc[level, "observations"])
                if level in country_counts.index
                else 0
            )
            psu_category = (
                int(country_counts.loc[level, "PSUs"])
                if level in country_counts.index
                else 0
            )

            row = {
                "sample": sample_name,
                "country": country,
                "treatment_code": level,
                "treatment_category": LEVELS[level],
                "N_no_treatment": n_control,
                "PSU_no_treatment": psu_control,
                "N_category": n_category,
                "PSU_category": psu_category,
                "both_categories_present": (
                    n_control > 0
                    and n_category > 0
                ),
                "at_least_2_PSU_each": (
                    psu_control >= 2
                    and psu_category >= 2
                ),
            }

            detail_rows.append(row)
            comparison_rows.append(row)

        comparison = pd.DataFrame(comparison_rows)
        category_sample = frame.loc[frame["treat_cat"].eq(level)]

        summary_rows.append(
            {
                "sample": sample_name,
                "treatment_category": LEVELS[level],
                "observations": len(category_sample),
                "PSUs": category_sample["_psu_id"].nunique(),
                "countries_with_category": (
                    category_sample["country_cat"].nunique()
                ),
                "countries_with_category_and_no_treatment": (
                    comparison["both_categories_present"].sum()
                ),
                "countries_with_at_least_2_PSU_each": (
                    comparison["at_least_2_PSU_each"].sum()
                ),
            }
        )

    return pd.DataFrame(detail_rows), pd.DataFrame(summary_rows)


In [24]:
hh_support_detail, hh_support_summary = support_by_country(
    hh_some_apos,
    "HH",
)

u5_support_detail, u5_support_summary = support_by_country(
    u5_apos,
    "U5",
)

support_detail = pd.concat(
    [hh_support_detail, u5_support_detail],
    ignore_index=True,
)

support_summary = pd.concat(
    [hh_support_summary, u5_support_summary],
    ignore_index=True,
)

support_detail.to_csv(
    OUT / "positivity_support_by_country.csv",
    index=False,
)

support_summary.to_csv(
    OUT / "positivity_support_summary.csv",
    index=False,
)

display(support_summary)


,sample,treatment_category,observations,PSUs,countries_with_category,countries_with_category_and_no_treatment,countries_with_at_least_2_PSU_each
0,HH,Boiling,7238,3070,25,25,24
1,HH,Chlorination/tablets,1458,1163,24,24,22
2,HH,Straining/settling,3816,2115,25,25,25
3,U5,Boiling,3216,1750,24,24,22
4,U5,Chlorination/tablets,838,521,21,21,19
5,U5,Straining/settling,2865,1145,25,25,25


### PSU-grouped sample splitting

Every PSU is assigned wholly to training or test. The audit also confirms that no household crosses the split, each observation appears exactly once in test per repetition, and every training fold contains all required treatment categories.


In [25]:
def grouped_sample_splitting(
    frame,
    treatment,
    n_rep,
    n_folds,
    seed,
):
    '''Construct repeated treatment-balanced folds grouped by PSU.'''

    y = frame[treatment].to_numpy(dtype=int)
    groups = frame["_psu_id"].to_numpy(dtype=int)
    households = frame["_hh_id"].to_numpy(dtype=int)
    required_levels = np.sort(np.unique(y))

    all_smpls = []
    all_smpls_cluster = []
    audit_rows = []

    for repetition in range(n_rep):
        selected = None

        for attempt in range(100):
            splitter = StratifiedGroupKFold(
                n_splits=n_folds,
                shuffle=True,
                random_state=(
                    seed
                    + repetition * 1_000
                    + attempt
                ),
            )

            candidate = list(
                splitter.split(
                    np.zeros(len(frame)),
                    y,
                    groups,
                )
            )

            valid = all(
                np.array_equal(
                    np.sort(np.unique(y[train])),
                    required_levels,
                )
                for train, _ in candidate
            )

            if valid:
                selected = candidate
                break

        if selected is None:
            raise ValueError(
                "No valid grouped split retained every treatment "
                "category in each training fold."
            )

        test_frequency = np.zeros(len(frame), dtype=int)
        repetition_smpls = []
        repetition_clusters = []

        for fold, (train, test) in enumerate(selected):
            train = np.asarray(train, dtype=int)
            test = np.asarray(test, dtype=int)

            train_psu = np.unique(groups[train])
            test_psu = np.unique(groups[test])
            train_households = np.unique(households[train])
            test_households = np.unique(households[test])

            if np.intersect1d(train_psu, test_psu).size:
                raise AssertionError("A PSU crosses training and test.")

            if np.intersect1d(
                train_households,
                test_households,
            ).size:
                raise AssertionError(
                    "A household crosses training and test."
                )

            test_frequency[test] += 1
            repetition_smpls.append((train, test))
            repetition_clusters.append(
                ([train_psu], [test_psu])
            )

            row = {
                "repetition": repetition,
                "fold": fold,
                "train_observations": len(train),
                "test_observations": len(test),
                "train_PSUs": len(train_psu),
                "test_PSUs": len(test_psu),
                "train_households": len(train_households),
                "test_households": len(test_households),
                "test_countries": frame.iloc[test][
                    "country_cat"
                ].nunique(),
            }

            for level in required_levels:
                row[f"test_share_{level}"] = np.mean(
                    y[test] == level
                )

            audit_rows.append(row)

        if not np.all(test_frequency == 1):
            raise AssertionError(
                "Test folds do not form a partition of the sample."
            )

        all_smpls.append(repetition_smpls)
        all_smpls_cluster.append(repetition_clusters)

    return (
        all_smpls,
        all_smpls_cluster,
        pd.DataFrame(audit_rows),
    )


In [26]:
hh_irm_smpls, hh_irm_cluster_smpls, hh_irm_audit = (
    grouped_sample_splitting(
        hh_some_irm,
        treatment="water_treatment",
        n_rep=IRM_REPS,
        n_folds=FOLDS,
        seed=SEED,
    )
)

hh_apos_smpls, hh_apos_cluster_smpls, hh_apos_audit = (
    grouped_sample_splitting(
        hh_some_apos,
        treatment="treat_cat",
        n_rep=APOS_REPS,
        n_folds=FOLDS,
        seed=SEED,
    )
)

u5_irm_smpls, u5_irm_cluster_smpls, u5_irm_audit = (
    grouped_sample_splitting(
        u5_irm,
        treatment="water_treatment",
        n_rep=IRM_REPS,
        n_folds=FOLDS,
        seed=SEED,
    )
)

u5_apos_smpls, u5_apos_cluster_smpls, u5_apos_audit = (
    grouped_sample_splitting(
        u5_apos,
        treatment="treat_cat",
        n_rep=APOS_REPS,
        n_folds=FOLDS,
        seed=SEED,
    )
)


In [27]:
fold_audit = pd.concat(
    [
        hh_irm_audit.assign(sample="HH", model="IRM"),
        hh_apos_audit.assign(sample="HH", model="APOS"),
        u5_irm_audit.assign(sample="U5", model="IRM"),
        u5_apos_audit.assign(sample="U5", model="APOS"),
    ],
    ignore_index=True,
)

fold_audit.to_csv(
    OUT / "fold_balance_grouped_convex_sl.csv",
    index=False,
)

display(fold_audit)


,repetition,fold,train_observations,test_observations,train_PSUs,test_PSUs,train_households,test_households,test_countries,test_share_0,test_share_1,sample,model,test_share_2,test_share_3
0,0,0,47580,11894,14948,3702,47580,11894,25,0.789474,0.210526,HH,IRM,NaN,NaN
1,0,1,47577,11897,14917,3733,47577,11897,25,0.789359,0.210641,HH,IRM,NaN,NaN
2,0,2,47577,11897,14915,3735,47577,11897,25,0.789443,0.210557,HH,IRM,NaN,NaN
3,0,3,47581,11893,14942,3708,47581,11893,25,0.789456,0.210544,HH,IRM,NaN,NaN
4,0,4,47581,11893,14878,3772,47581,11893,25,0.789456,0.210544,HH,IRM,NaN,NaN
5,1,0,47578,11896,14930,3720,47578,11896,25,0.789341,0.210659,HH,IRM,NaN,NaN
6,1,1,47580,11894,14913,3737,47580,11894,25,0.789390,0.210610,HH,IRM,NaN,NaN
7,1,2,47580,11894,14915,3735,47580,11894,25,0.789474,0.210526,HH,IRM,NaN,NaN
8,1,3,47579,11895,14928,3722,47579,11895,25,0.789491,0.210509,HH,IRM,NaN,NaN
9,1,4,47579,11895,14914,3736,47579,11895,25,0.789491,0.210509,HH,IRM,NaN,NaN


### Checkpoint utilities

A checkpoint is loaded only when its data, controls, folds, learners, trimming rule, and package versions match the current specification. Otherwise the model is estimated and the checkpoint is replaced.


In [28]:
def write_pickle(path, value):
    '''Write a pickle atomically.'''

    temporary = path.with_suffix(path.suffix + ".tmp")

    with temporary.open("wb") as file:
        pickle.dump(
            value,
            file,
            protocol=pickle.HIGHEST_PROTOCOL,
        )
        file.flush()
        os.fsync(file.fileno())

    os.replace(temporary, path)


def read_checkpoint(path, signature):
    '''Return a compatible cached object, otherwise None.'''

    if not path.is_file():
        return None

    try:
        with path.open("rb") as file:
            cache = pickle.load(file)
    except Exception:
        tqdm.write(f"Unreadable checkpoint; rebuilding {path.name}")
        return None

    if cache.get("signature") != signature:
        tqdm.write(f"Checkpoint changed; rebuilding {path.name}")
        return None

    tqdm.write(f"Loaded {path.name}")
    return cache["value"]


def cached_value(path, signature, builder):
    '''Load a compatible checkpoint or compute and save it.'''

    cached = read_checkpoint(path, signature)

    if cached is not None:
        return cached

    value = builder()
    write_pickle(
        path,
        {
            "signature": signature,
            "value": value,
        },
    )
    tqdm.write(f"Saved {path.name}")

    return value


In [29]:
def frame_signature(frame, columns):
    '''Hash the rows and columns relevant to one fit.'''

    return joblib_hash(frame[columns])


def fit_signature(
    kind,
    frame,
    outcome,
    treatment,
    x_cols,
    ml_g,
    ml_m,
    smpls,
):
    '''Identify one exact DDML specification.'''

    settings = {
        "kind": kind,
        "data": frame_signature(
            frame,
            [
                "_row_id",
                "country_cat",
                "_psu_id",
                "_hh_id",
                outcome,
                treatment,
                *x_cols,
            ],
        ),
        "outcome": outcome,
        "treatment": treatment,
        "controls": x_cols,
        "ml_g": joblib_hash(ml_g),
        "ml_m": joblib_hash(ml_m),
        "sample_splitting": joblib_hash(smpls),
        "trimming": TRIM,
        "seed": SEED,
        "doubleml": dml.__version__,
        "sklearn": sklearn.__version__,
    }

    return joblib_hash(settings)


### Raw out-of-fold propensity scores

These fits estimate only the treatment nuisance functions. The raw predictions diagnose positivity; their clipped counterparts are retained separately because clipping is applied only when the AIPW score is calculated.


In [30]:
def raw_oof_propensities(
    frame,
    treatment,
    x_cols,
    smpls,
    levels,
    sample_name,
    model_name,
):
    '''Generate raw and clipped propensity scores on outer test folds.'''

    X = frame[x_cols].to_numpy(dtype=float)
    observed = frame[treatment].to_numpy(dtype=int)

    prediction_rows = []
    weight_rows = []

    for repetition, repetition_smpls in enumerate(smpls):
        for level in levels:
            predictions = np.full(len(frame), np.nan)

            for fold, (train, test) in enumerate(repetition_smpls):
                learner = clone(ML_M)
                target = (observed[train] == level).astype(int)

                learner.fit(X[train], target)
                predictions[test] = learner.predict_proba(
                    X[test]
                )[:, 1]

                for name, weight in zip(
                    learner.estimator_names_,
                    learner.weights_,
                ):
                    weight_rows.append(
                        {
                            "sample": sample_name,
                            "model": model_name,
                            "treatment_code": level,
                            "treatment_category": (
                                LEVELS.get(level, str(level))
                            ),
                            "repetition": repetition,
                            "fold": fold,
                            "learner": name,
                            "weight": weight,
                        }
                    )

            if np.isnan(predictions).any():
                raise RuntimeError("Incomplete outer propensity predictions.")

            clipped = np.clip(predictions, TRIM, 1 - TRIM)

            for position, (raw, adjusted) in enumerate(
                zip(predictions, clipped)
            ):
                prediction_rows.append(
                    {
                        "sample": sample_name,
                        "model": model_name,
                        "row_id": frame["_row_id"].iat[position],
                        "country": frame["country_cat"].iat[position],
                        "PSU": frame["_psu_id"].iat[position],
                        "observed_treatment": frame[treatment].iat[position],
                        "treatment_code": level,
                        "treatment_category": (
                            LEVELS.get(level, str(level))
                        ),
                        "repetition": repetition,
                        "propensity_raw": raw,
                        "propensity_clipped": adjusted,
                    }
                )

    return pd.DataFrame(prediction_rows), pd.DataFrame(weight_rows)


In [31]:
def propensity_checkpoint(
    path,
    frame,
    treatment,
    x_cols,
    smpls,
    levels,
    sample_name,
    model_name,
):
    '''Load or compute raw OOF propensity diagnostics.'''

    signature = joblib_hash(
        {
            "data": frame_signature(
                frame,
                [
                    "_row_id",
                    "country_cat",
                    "_psu_id",
                    treatment,
                    *x_cols,
                ],
            ),
            "treatment": treatment,
            "controls": x_cols,
            "splits": smpls,
            "levels": levels,
            "learner": ML_M,
            "trim": TRIM,
            "sample": sample_name,
            "model": model_name,
        }
    )

    return cached_value(
        path,
        signature,
        lambda: raw_oof_propensities(
            frame,
            treatment,
            x_cols,
            smpls,
            levels,
            sample_name,
            model_name,
        ),
    )


In [ ]:
hh_irm_oof, hh_irm_weights = propensity_checkpoint(
    OOF_MODELS / "hh_irm.pkl",
    hh_some_irm,
    "water_treatment",
    hh_irm_x,
    hh_irm_smpls,
    levels=[1],
    sample_name="HH",
    model_name="Any recognized treatment",
)

hh_apos_oof, hh_apos_weights = propensity_checkpoint(
    OOF_MODELS / "hh_apos.pkl",
    hh_some_apos,
    "treat_cat",
    hh_apos_x,
    hh_apos_smpls,
    levels=list(LEVELS),
    sample_name="HH",
    model_name="Treatment categories",
)

u5_irm_oof, u5_irm_weights = propensity_checkpoint(
    OOF_MODELS / "u5_irm.pkl",
    u5_irm,
    "water_treatment",
    u5_irm_x,
    u5_irm_smpls,
    levels=[1],
    sample_name="U5",
    model_name="Any recognized treatment",
)

u5_apos_oof, u5_apos_weights = propensity_checkpoint(
    OOF_MODELS / "u5_apos.pkl",
    u5_apos,
    "treat_cat",
    u5_apos_x,
    u5_apos_smpls,
    levels=list(LEVELS),
    sample_name="U5",
    model_name="Treatment categories",
)


In [ ]:
propensity_oof = pd.concat(
    [
        hh_irm_oof,
        hh_apos_oof,
        u5_irm_oof,
        u5_apos_oof,
    ],
    ignore_index=True,
)

propensity_weights = pd.concat(
    [
        hh_irm_weights,
        hh_apos_weights,
        u5_irm_weights,
        u5_apos_weights,
    ],
    ignore_index=True,
)

propensity_oof.to_csv(
    OUT / "propensity_oof_raw_and_clipped.csv",
    index=False,
)

propensity_weights.to_csv(
    OUT / "super_learner_propensity_weights.csv",
    index=False,
)


### Propensity summary


In [ ]:
def propensity_summary(oof):
    '''Create the compact positivity table.'''

    rows = []

    group_columns = [
        "sample",
        "model",
        "treatment_code",
        "treatment_category",
    ]

    for keys, group in oof.groupby(group_columns):
        values = group["propensity_raw"]
        quantiles = values.quantile(
            [0.01, 0.05, 0.50, 0.95, 0.99]
        )

        rows.append(
            {
                "sample": keys[0],
                "model": keys[1],
                "treatment_code": keys[2],
                "treatment_category": keys[3],
                "p01": quantiles.loc[0.01],
                "p05": quantiles.loc[0.05],
                "median": quantiles.loc[0.50],
                "p95": quantiles.loc[0.95],
                "p99": quantiles.loc[0.99],
                "below_0.01_percent": 100 * values.lt(0.01).mean(),
                "above_0.99_percent": 100 * values.gt(0.99).mean(),
                "clipped_percent": 100 * (
                    group["propensity_raw"]
                    .ne(group["propensity_clipped"])
                    .mean()
                ),
            }
        )

    return pd.DataFrame(rows)


propensity_table = propensity_summary(propensity_oof)

propensity_table.to_csv(
    OUT / "propensity_oof_summary.csv",
    index=False,
)

display(propensity_table)


### Propensity plots

The figures use raw OOF predictions. The vertical reference lines show the clipping threshold used later in the AIPW score.


In [ ]:
def plot_propensity_ecdf(oof, sample_name, model_name, level):
    '''Plot one empirical CDF for a treatment propensity.'''

    data = oof.loc[
        oof["sample"].eq(sample_name)
        & oof["model"].eq(model_name)
        & oof["treatment_code"].eq(level)
    ]

    values = np.sort(data["propensity_raw"].to_numpy())
    cumulative = np.arange(1, len(values) + 1) / len(values)

    plt.figure(figsize=(7, 4.5))
    plt.plot(values, cumulative)
    plt.axvline(TRIM, linestyle="--")
    plt.axvline(1 - TRIM, linestyle="--")
    plt.xlabel("Raw out-of-fold propensity")
    plt.ylabel("Empirical cumulative probability")
    plt.title(
        f"{sample_name}: "
        f"{LEVELS.get(level, model_name)}"
    )
    plt.tight_layout()

    filename = (
        f"propensity_ecdf_{sample_name.lower()}_"
        f"{model_name.lower().replace(' ', '_')}_{level}.png"
    )

    plt.savefig(FIGS / filename, dpi=200)
    plt.show()


In [ ]:
for sample_name in ["HH", "U5"]:
    for level in [1, 2, 3]:
        plot_propensity_ecdf(
            propensity_oof,
            sample_name,
            "Treatment categories",
            level,
        )


### Diagnostic checkpoint

At this point the notebook has checked outcome coding, treatment coding, PSU structure, observed category support, fold integrity, and raw OOF propensities. No causal effect has yet been estimated.


In [ ]:
diagnostic_summary = pd.Series(
    {
        "other_only_removed_HH": hh_cleaning["other_only_rows_dropped"],
        "other_only_removed_U5": u5_cleaning["other_only_rows_dropped"],
        "HH_PSUs": hh_some_irm["_psu_id"].nunique(),
        "U5_PSUs": u5_irm["_psu_id"].nunique(),
        "HH_fold_rows": len(hh_irm_audit) + len(hh_apos_audit),
        "U5_fold_rows": len(u5_irm_audit) + len(u5_apos_audit),
        "raw_propensity_rows": len(propensity_oof),
    },
    name="value",
)

display(diagnostic_summary)


## Estimation

### DoubleML fitting functions


In [ ]:
def propensity_processing_kwargs():
    '''Use the current DoubleML propensity-processing interface when available.'''

    if PSProcessorConfig is not None:
        return {
            "ps_processor_config": PSProcessorConfig(
                clipping_threshold=TRIM,
            )
        }

    return {
        "trimming_rule": "truncate",
        "trimming_threshold": TRIM,
    }


In [ ]:
def fit_irm_model(
    frame,
    outcome,
    treatment,
    x_cols,
    smpls,
    cluster_smpls,
    ml_g,
    ml_m,
):
    '''Fit one clustered IRM model with externally supplied folds.'''

    data = dml.DoubleMLData(
        frame,
        y_col=outcome,
        d_cols=treatment,
        x_cols=x_cols,
        cluster_cols="_psu_id",
    )

    model = dml.DoubleMLIRM(
        data,
        ml_g=clone(ml_g),
        ml_m=clone(ml_m),
        score="ATE",
        draw_sample_splitting=False,
        **propensity_processing_kwargs(),
    )

    model.set_sample_splitting(
        smpls,
        cluster_smpls,
    )

    model.fit(
        n_jobs_cv=IRM_WORKERS,
        store_predictions=True,
        store_models=False,
    )

    return model


In [ ]:
def fit_apos_model(
    frame,
    outcome,
    treatment,
    x_cols,
    smpls,
    cluster_smpls,
    ml_g,
    ml_m,
):
    '''Fit average potential outcomes and contrasts to no treatment.'''

    data = dml.DoubleMLData(
        frame,
        y_col=outcome,
        d_cols=treatment,
        x_cols=x_cols,
        cluster_cols="_psu_id",
    )

    model = dml.DoubleMLAPOS(
        data,
        ml_g=clone(ml_g),
        ml_m=clone(ml_m),
        treatment_levels=list(LEVELS),
        draw_sample_splitting=False,
        **propensity_processing_kwargs(),
    )

    model.set_sample_splitting(
        smpls,
        cluster_smpls,
    )

    model.fit(
        n_jobs_models=APOS_WORKERS,
        n_jobs_cv=1,
        store_predictions=True,
        store_models=False,
    )

    contrast = model.causal_contrast(reference_levels=0)

    return {
        "model": model,
        "contrast": contrast,
    }


In [ ]:
def cached_irm(
    path,
    frame,
    outcome,
    treatment,
    x_cols,
    smpls,
    cluster_smpls,
    ml_g=ML_G,
    ml_m=ML_M,
):
    '''Load or estimate one IRM checkpoint.'''

    signature = fit_signature(
        "IRM",
        frame,
        outcome,
        treatment,
        x_cols,
        ml_g,
        ml_m,
        smpls,
    )

    return cached_value(
        path,
        signature,
        lambda: fit_irm_model(
            frame,
            outcome,
            treatment,
            x_cols,
            smpls,
            cluster_smpls,
            ml_g,
            ml_m,
        ),
    )


def cached_apos(
    path,
    frame,
    outcome,
    treatment,
    x_cols,
    smpls,
    cluster_smpls,
    ml_g=ML_G,
    ml_m=ML_M,
):
    '''Load or estimate one APOS checkpoint.'''

    signature = fit_signature(
        "APOS",
        frame,
        outcome,
        treatment,
        x_cols,
        ml_g,
        ml_m,
        smpls,
    )

    return cached_value(
        path,
        signature,
        lambda: fit_apos_model(
            frame,
            outcome,
            treatment,
            x_cols,
            smpls,
            cluster_smpls,
            ml_g,
            ml_m,
        ),
    )


### Any recognized treatment


In [ ]:
irm_models = {
    "SomeRiskHome": cached_irm(
        MODELS / "irm_hh_some_risk.pkl",
        hh_some_irm,
        "SomeRiskHome",
        "water_treatment",
        hh_irm_x,
        hh_irm_smpls,
        hh_irm_cluster_smpls,
    ),
    "VeryHighRiskHome": cached_irm(
        MODELS / "irm_hh_very_high_risk.pkl",
        hh_vhigh_irm,
        "VeryHighRiskHome",
        "water_treatment",
        hh_vhigh_irm_x,
        hh_irm_smpls,
        hh_irm_cluster_smpls,
    ),
    "diarrhea": cached_irm(
        MODELS / "irm_u5_diarrhea.pkl",
        u5_irm,
        "diarrhea",
        "water_treatment",
        u5_irm_x,
        u5_irm_smpls,
        u5_irm_cluster_smpls,
    ),
}


### Treatment categories


In [ ]:
apos_models = {
    "SomeRiskHome": cached_apos(
        MODELS / "apos_hh_some_risk.pkl",
        hh_some_apos,
        "SomeRiskHome",
        "treat_cat",
        hh_apos_x,
        hh_apos_smpls,
        hh_apos_cluster_smpls,
    ),
    "VeryHighRiskHome": cached_apos(
        MODELS / "apos_hh_very_high_risk.pkl",
        hh_vhigh_apos,
        "VeryHighRiskHome",
        "treat_cat",
        hh_vhigh_apos_x,
        hh_apos_smpls,
        hh_apos_cluster_smpls,
    ),
    "diarrhea": cached_apos(
        MODELS / "apos_u5_diarrhea.pkl",
        u5_apos,
        "diarrhea",
        "treat_cat",
        u5_apos_x,
        u5_apos_smpls,
        u5_apos_cluster_smpls,
    ),
}


## Results

### Main estimates


In [ ]:
def summary_value(row, names):
    '''Read one value despite minor DoubleML column-name differences.'''

    for name in names:
        if name in row.index:
            return row[name]

    return np.nan


def extract_irm_result(outcome, model):
    '''Convert one IRM summary to a tidy row.'''

    row = model.summary.iloc[0]

    return {
        "model": "Any recognized treatment",
        "outcome": outcome,
        "outcome_label": OUTCOME_LABELS[outcome],
        "comparison": "Any recognized treatment vs no treatment",
        "coefficient": summary_value(row, ["coef"]),
        "standard_error": summary_value(row, ["std err", "std_error"]),
        "p_value": summary_value(row, ["P>|t|", "pval"]),
        "ci_low": summary_value(row, ["2.5 %", "2.5%"]),
        "ci_high": summary_value(row, ["97.5 %", "97.5%"]),
    }


def extract_contrast_results(outcome, contrast):
    '''Convert APOS contrasts to tidy rows.'''

    rows = []

    for index, result in contrast.summary.iterrows():
        level = int(float(str(index).split(" vs ")[0]))

        rows.append(
            {
                "model": "Treatment categories",
                "outcome": outcome,
                "outcome_label": OUTCOME_LABELS[outcome],
                "comparison": f"{LEVELS[level]} vs no treatment",
                "treatment_code": level,
                "treatment_category": LEVELS[level],
                "coefficient": summary_value(result, ["coef"]),
                "standard_error": summary_value(
                    result,
                    ["std err", "std_error"],
                ),
                "p_value": summary_value(result, ["P>|t|", "pval"]),
                "ci_low": summary_value(result, ["2.5 %", "2.5%"]),
                "ci_high": summary_value(result, ["97.5 %", "97.5%"]),
            }
        )

    return rows


In [ ]:
main_rows = []

for outcome, model in irm_models.items():
    main_rows.append(extract_irm_result(outcome, model))

for outcome, bundle in apos_models.items():
    main_rows.extend(
        extract_contrast_results(
            outcome,
            bundle["contrast"],
        )
    )

main_results = pd.DataFrame(main_rows)

main_results.to_csv(
    OUT / "results_main_grouped_convex_sl.csv",
    index=False,
)

display(main_results)


### Convex propensity weights

These are diagnostic weights for the treatment nuisance learner. They are not inverse-probability weights and do not enter the publication table.


In [ ]:
weight_summary = (
    propensity_weights.groupby(
        [
            "sample",
            "model",
            "treatment_category",
            "learner",
        ]
    )["weight"]
    .agg(["mean", "median", "std", "min", "max"])
    .reset_index()
)

weight_summary.to_csv(
    OUT / "super_learner_propensity_weight_summary.csv",
    index=False,
)

display(weight_summary)


## Robustness

### Support-restricted comparisons

These estimates are calculated separately for each treatment category. The basic restriction retains countries in which both categories are observed. The stricter restriction additionally requires at least two PSU in each category. These analyses change the target population and are therefore reported as robustness checks rather than as the principal estimates.


In [ ]:
def eligible_countries(detail, level, minimum_psu):
    '''Return countries satisfying one observed-support rule.'''

    subset = detail.loc[detail["treatment_code"].eq(level)]

    if minimum_psu == 1:
        keep = subset["both_categories_present"]
    elif minimum_psu == 2:
        keep = subset["at_least_2_PSU_each"]
    else:
        raise ValueError("minimum_psu must be 1 or 2.")

    return set(subset.loc[keep, "country"])


def pairwise_frame(
    df,
    outcome,
    level,
    countries,
    child=False,
):
    '''Build a treatment-category versus no-treatment sample.'''

    sample = df.loc[
        df["country_cat"].isin(countries)
        & df["treat_cat"].isin([0, level])
    ].copy()

    sample["pair_treatment"] = sample["treat_cat"].eq(level).astype("Int8")

    return analysis_frame(
        sample,
        outcome=outcome,
        treatment="pair_treatment",
        child=child,
    )


In [ ]:
def support_robustness_fit(
    sample_name,
    df,
    outcome,
    level,
    detail,
    minimum_psu,
    child=False,
):
    '''Fit one pairwise support-restricted IRM model.'''

    countries = eligible_countries(
        detail,
        level,
        minimum_psu,
    )

    frame, x_cols = pairwise_frame(
        df,
        outcome,
        level,
        countries,
        child=child,
    )

    smpls, cluster_smpls, _ = grouped_sample_splitting(
        frame,
        treatment="pair_treatment",
        n_rep=ROBUSTNESS_REPS,
        n_folds=FOLDS,
        seed=SEED + 10_000 + level + minimum_psu,
    )

    filename = (
        f"{sample_name.lower()}_{outcome}_"
        f"level_{level}_psu_{minimum_psu}.pkl"
    )

    model = cached_irm(
        SUPPORT_MODELS / filename,
        frame,
        outcome,
        "pair_treatment",
        x_cols,
        smpls,
        cluster_smpls,
    )

    result = extract_irm_result(outcome, model)
    result.update(
        {
            "sample": sample_name,
            "treatment_code": level,
            "treatment_category": LEVELS[level],
            "comparison": f"{LEVELS[level]} vs no treatment",
            "support_rule": (
                "Both categories present"
                if minimum_psu == 1
                else "At least two PSU in each category"
            ),
            "countries": len(countries),
            "observations": len(frame),
            "PSUs": frame["_psu_id"].nunique(),
        }
    )

    return result


In [ ]:
support_rows = []

support_specs = [
    ("HH", hh, "SomeRiskHome", hh_support_detail, False),
    ("HH", hh, "VeryHighRiskHome", hh_support_detail, False),
    ("U5", u5, "diarrhea", u5_support_detail, True),
]

for sample_name, data, outcome, detail, child in support_specs:
    for level in [1, 2, 3]:
        for minimum_psu in [1, 2]:
            try:
                result = support_robustness_fit(
                    sample_name,
                    data,
                    outcome,
                    level,
                    detail,
                    minimum_psu,
                    child=child,
                )
            except ValueError as error:
                result = {
                    "sample": sample_name,
                    "outcome": outcome,
                    "treatment_code": level,
                    "treatment_category": LEVELS[level],
                    "support_rule": minimum_psu,
                    "status": str(error),
                }

            support_rows.append(result)

support_results = pd.DataFrame(support_rows)

support_results.to_csv(
    OUT / "results_support_restricted.csv",
    index=False,
)

display(support_results)


### Leave-one-country-out

Each country is excluded in turn. Country indicators, grouped folds, and nuisance functions are rebuilt after every exclusion. Only the convex Super Learner specification is used.


In [ ]:
def loco_irm_fit(
    sample_name,
    df,
    outcome,
    excluded_country,
    child=False,
):
    '''Refit the binary treatment model after excluding one country.'''

    reduced = df.loc[
        ~df["country_cat"].eq(excluded_country)
    ].copy()

    frame, x_cols = analysis_frame(
        reduced,
        outcome,
        "water_treatment",
        child=child,
    )

    smpls, cluster_smpls, _ = grouped_sample_splitting(
        frame,
        "water_treatment",
        LOCO_REPS,
        FOLDS,
        SEED + 20_000 + int(joblib_hash(excluded_country)[:8], 16),
    )

    filename = (
        f"irm_{sample_name.lower()}_{outcome}_"
        f"without_{excluded_country}.pkl"
    )

    model = cached_irm(
        LOCO_MODELS / filename,
        frame,
        outcome,
        "water_treatment",
        x_cols,
        smpls,
        cluster_smpls,
    )

    result = extract_irm_result(outcome, model)
    result.update(
        {
            "sample": sample_name,
            "excluded_country": excluded_country,
            "observations": len(frame),
            "PSUs": frame["_psu_id"].nunique(),
            "households": frame["_hh_id"].nunique(),
        }
    )

    return result


In [ ]:
def loco_apos_fit(
    sample_name,
    df,
    outcome,
    excluded_country,
    child=False,
):
    '''Refit treatment-category contrasts after excluding one country.'''

    reduced = df.loc[
        ~df["country_cat"].eq(excluded_country)
    ].copy()

    frame, x_cols = analysis_frame(
        reduced,
        outcome,
        "treat_cat",
        child=child,
        allowed_levels=list(LEVELS),
    )

    if set(frame["treat_cat"].unique()) != set(LEVELS):
        raise ValueError("At least one treatment category disappears.")

    smpls, cluster_smpls, _ = grouped_sample_splitting(
        frame,
        "treat_cat",
        LOCO_REPS,
        FOLDS,
        SEED + 30_000 + int(joblib_hash(excluded_country)[:8], 16),
    )

    filename = (
        f"apos_{sample_name.lower()}_{outcome}_"
        f"without_{excluded_country}.pkl"
    )

    bundle = cached_apos(
        LOCO_MODELS / filename,
        frame,
        outcome,
        "treat_cat",
        x_cols,
        smpls,
        cluster_smpls,
    )

    rows = extract_contrast_results(
        outcome,
        bundle["contrast"],
    )

    for row in rows:
        row.update(
            {
                "sample": sample_name,
                "excluded_country": excluded_country,
                "observations": len(frame),
                "PSUs": frame["_psu_id"].nunique(),
                "households": frame["_hh_id"].nunique(),
            }
        )

    return rows


In [ ]:
loco_rows = []

loco_specs = [
    ("HH", hh, "SomeRiskHome", False),
    ("HH", hh, "VeryHighRiskHome", False),
    ("U5", u5, "diarrhea", True),
]

for sample_name, data, outcome, child in loco_specs:
    countries = sorted(data["country_cat"].dropna().unique())

    for country in tqdm(
        countries,
        desc=f"LOCO {sample_name} {outcome}",
        leave=False,
    ):
        try:
            loco_rows.append(
                loco_irm_fit(
                    sample_name,
                    data,
                    outcome,
                    country,
                    child=child,
                )
            )

            loco_rows.extend(
                loco_apos_fit(
                    sample_name,
                    data,
                    outcome,
                    country,
                    child=child,
                )
            )
        except ValueError as error:
            loco_rows.append(
                {
                    "sample": sample_name,
                    "outcome": outcome,
                    "excluded_country": country,
                    "status": str(error),
                }
            )

loco_results = pd.DataFrame(loco_rows)

loco_results.to_csv(
    OUT / "results_leave_one_country_out.csv",
    index=False,
)

display(loco_results.head())


### LOCO influence relative to the full estimate


In [ ]:
full_lookup = main_results.set_index(
    ["outcome", "comparison"]
)[["coefficient", "standard_error"]]

loco_complete = loco_results.dropna(
    subset=["coefficient"]
).copy()

loco_complete = loco_complete.join(
    full_lookup,
    on=["outcome", "comparison"],
    rsuffix="_full",
)

loco_complete["change_from_full"] = (
    loco_complete["coefficient"]
    - loco_complete["coefficient_full"]
)

loco_complete["change_in_full_SE"] = (
    loco_complete["change_from_full"]
    / loco_complete["standard_error_full"]
)

loco_complete.to_csv(
    OUT / "results_leave_one_country_out_influence.csv",
    index=False,
)

display(
    loco_complete.sort_values(
        "change_in_full_SE",
        key=lambda values: values.abs(),
        ascending=False,
    ).head(20)
)


## Output manifest


In [ ]:
output_manifest = {
    "cleaning_diagnostics": str(
        OUT / "cleaning_and_identifier_diagnostics.csv"
    ),
    "treatment_frequencies": str(
        OUT / "treatment_category_frequencies.csv"
    ),
    "PSU_structure": str(
        OUT / "psu_structure_summary.csv"
    ),
    "support_by_country": str(
        OUT / "positivity_support_by_country.csv"
    ),
    "support_summary": str(
        OUT / "positivity_support_summary.csv"
    ),
    "fold_audit": str(
        OUT / "fold_balance_grouped_convex_sl.csv"
    ),
    "raw_propensities": str(
        OUT / "propensity_oof_raw_and_clipped.csv"
    ),
    "propensity_summary": str(
        OUT / "propensity_oof_summary.csv"
    ),
    "main_results": str(
        OUT / "results_main_grouped_convex_sl.csv"
    ),
    "support_restricted_results": str(
        OUT / "results_support_restricted.csv"
    ),
    "LOCO_results": str(
        OUT / "results_leave_one_country_out.csv"
    ),
}

manifest_path = OUT / "grouped_convex_sl_manifest.json"
manifest_path.write_text(
    json.dumps(output_manifest, indent=2),
    encoding="utf-8",
)

display(pd.Series(output_manifest, name="path"))
